# 01. GCR Baseline Reproduction

**Purpose:** Reproduce the Graph-Constrained Reasoning (GCR) baseline on WebQSP and CWQ.

**Output:** Saved predictions JSONL at `results/GenPaths/{dataset}/GCR-{model}/test/{config}/predictions.jsonl`

**Reference:** Luo et al. (2025) — GCR achieves 92.6 Hits@1 on WebQSP, 75.8 on CWQ.

Based on: `workflow/predict_paths_and_answers.py` + `scripts/graph_constrained_decoding.sh`

**Note:** This is the unmodified GCR baseline, used to establish the reference SIR and Hits@1 for comparison with DCA-Trie variants in notebooks 02-04.

In [1]:
# @title 1. Install & Setup
import sys, os, torch, json, math
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
from tqdm import tqdm
from datasets import load_dataset
from multiprocessing import Pool
from functools import partial

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
has_a100 = "A100" in gpu_name
print(f"GPU: {gpu_name}  |  VRAM: {gpu_mem:.1f} GB")

!pip install -q transformers==4.44.2 accelerate peft deepspeed tiktoken datasets python-dotenv marisa-trie scikit-learn bitsandbytes trl sentencepiece protobuf wandb networkx sentence-transformers

if not os.path.exists("graph-constrained-reasoning"):
    !git clone https://github.com/rmanluo/graph-constrained-reasoning.git
%cd graph-constrained-reasoning
sys.path.insert(0, '.')
from src.llms import get_registed_model

GPU: NVIDIA A100-SXM4-40GB  |  VRAM: 42.4 GB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 97.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.5 MB/s eta 0:00:00
Cloning into 'graph-constrained-reasoning'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (35/35

In [4]:
import argparse
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

# @title 2. Configuration
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MODEL
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL_PATH = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
MODEL_NAME = "rmanluo/GCR-Meta-Llama-3.1-8B-Instruct"
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DATASETS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_PATH = "rmanluo"
DATASETS = ["RoG-webqsp", "RoG-cwq"]
SPLIT = "test"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# DECODING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INDEX_LEN = 2
K = 5
GEN_MODE = "group-beam"
PROMPT_MODE = "zero-shot"
MAX_NEW_TOKENS = 256
N_PROC = 1
ATTN_IMPL = "sdpa" # Changed from "flash_attention_2" if has_a100 else "sdpa"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# SAMPLING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MAX_SAMPLES = 100          # set to None for full dataset

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# OUTPUT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PREDICT_PATH = "results/GenPaths"
FORCE = True

print(f"Model: {MODEL_PATH}")
print(f"Attention: {ATTN_IMPL}")
print(f"Datasets: {DATASETS}")
print(f"Beam width k: {K}")
print(f"Max path length: {INDEX_LEN}")
print(f"Max samples: {MAX_SAMPLES or 'full'}")

Model: rmanluo/GCR-Meta-Llama-3.1-8B-Instruct
Attention: sdpa
Datasets: ['RoG-webqsp', 'RoG-cwq']
Beam width k: 5
Max path length: 2
Max samples: 100


In [5]:
# @title 3. Load Model (shared across datasets)
import argparse
from src.qa_prompt_builder import PathGenerationWithAnswerPromptBuilder

parser = argparse.ArgumentParser()
LLM = get_registed_model(MODEL_NAME)
LLM.add_args(parser)
args = parser.parse_args([])
args.model_path = MODEL_PATH
args.model_name = MODEL_NAME
args.k = K
args.generation_mode = GEN_MODE
args.attn_implementation = ATTN_IMPL
args.max_new_tokens = MAX_NEW_TOKENS
args.maximun_token = 4096

print("Loading model (this takes 1-2 minutes)...")
model = LLM(args)
model.prepare_for_inference()
# Force-clear sampling params on the underlying model config to suppress warnings
model.generation_cfg.temperature = None
model.generation_cfg.top_p = None
model.generation_cfg.top_k = None
model.model.generation_config.temperature = None
model.model.generation_config.top_p = None
model.model.generation_config.top_k = None
print("Model loaded.")

import src.utils as utils

Loading model (this takes 1-2 minutes)...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded.


In [6]:
# @title 4. Run GCR Baseline on Selected Dataset
def get_output_file(path, force=False):
    if not os.path.exists(path) or force:
        fout = open(path, "w")
        return fout, []
    else:
        with open(path, "r") as f:
            processed = [json.loads(line)["id"] for line in f]
        fout = open(path, "a")
        return fout, processed

def predict_one(data, processed_list, input_builder, model):
    qid = data["id"]
    if qid in processed_list:
        return None
    input_query, ground_paths, trie = input_builder.process_input(data)
    if trie is None:
        return None
    start_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_START_TOKEN)
    end_token_ids = model.tokenizer.convert_tokens_to_ids(input_builder.PATH_END_TOKEN)
    llm_input = model.prepare_model_prompt(input_query)
    prediction = model.generate_sentence(
        llm_input, trie,
        start_token_ids=start_token_ids,
        end_token_ids=end_token_ids,
        enable_constrained_by_default=False,
    )
    if prediction is None:
        return None
    return {
        "id": qid,
        "question": data["question"],
        "prediction": prediction,
        "ground_truth": data["answer"],
        "ground_truth_paths": ground_paths,
        "input": input_query,
    }

SELECTED_DATASET = "RoG-webqsp"  # or "RoG-cwq"

print(f"Processing {SELECTED_DATASET}...")
dataset = load_dataset(f"{DATA_PATH}/{SELECTED_DATASET}", split=SPLIT)
print(f"Loaded {len(dataset)} questions")

if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
    print(f"Subsampled to {len(dataset)} questions")

input_builder = PathGenerationWithAnswerPromptBuilder(
    model.tokenizer, PROMPT_MODE, index_path_length=INDEX_LEN
)

postfix = f"{PROMPT_MODE}-{GEN_MODE}-k{K}-index_len{INDEX_LEN}"
data_name = SELECTED_DATASET
model_short = MODEL_PATH.split("/")[-1]
output_dir = os.path.join(PREDICT_PATH, data_name, model_short, SPLIT, postfix)
os.makedirs(output_dir, exist_ok=True)
print(f"Output: {output_dir}")

fout, processed = get_output_file(os.path.join(output_dir, "predictions.jsonl"), force=FORCE)
print(f"Already processed: {len(processed)} questions")

for data in tqdm(dataset, desc="Generating"):
    res = predict_one(data, processed, input_builder, model)
    if res is not None:
        fout.write(json.dumps(res) + "\n")
        fout.flush()
fout.close()

from src.utils.qa_utils import eval_path_result_w_ans
print("\n=== Evaluation ===")
eval_path_result_w_ans(os.path.join(output_dir, "predictions.jsonl"))

Processing RoG-webqsp...


README.md:   0%|          | 0.00/900 [00:00<?, ?B/s]

data/train-00000-of-00002-d810a36ed97bc2(…):   0%|          | 0.00/154M [00:00<?, ?B/s]

data/train-00001-of-00002-e53244e71082a3(…):   0%|          | 0.00/155M [00:00<?, ?B/s]

data/validation-00000-of-00001-6ee6adc5b(…):   0%|          | 0.00/24.3M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/246 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1628 [00:00<?, ? examples/s]

Loaded 1628 questions
Subsampled to 100 questions
Output: results/GenPaths/RoG-webqsp/GCR-Meta-Llama-3.1-8B-Instruct/test/zero-shot-group-beam-k5-index_len2
Already processed: 0 questions


Generating: 100%|██████████| 100/100 [07:14<00:00,  4.34s/it]



=== Evaluation ===
Accuracy: 70.77287907163698 Hit: 89.0 F1: 62.77118817984933 Precision: 69.48333333333333 Recall: 70.61289592410785 Path F1: 60.53325638173157 Path Precision: 69.08333333333333 Path Recall: 65.40358395049124 Path Answer F1: 62.89740315576711 Path Answer Precision: 69.48333333333333 Path Answer Recall: 70.76696191187366


In [7]:
# @title 5. Compare with Published Results
print("""
=== GCR Published Results (Luo et al. 2025) ===

                    WebQSP     CWQ
  GCR (Llama-3.1-8B)
    Hits@1           92.6      75.8
    F1               89.8      69.4
    Faithfulness    100%      100%

Your reproduction should be within 1-2% of these.
""")


=== GCR Published Results (Luo et al. 2025) ===

                    WebQSP     CWQ
  GCR (Llama-3.1-8B)
    Hits@1           92.6      75.8
    F1               89.8      69.4
    Faithfulness    100%      100%

Your reproduction should be within 1-2% of these.

